# Konoros-Sec v1.1 — Colab Train & Test (defensive only)
Repo đã có sẵn data thật: `data/raw/so_all.jsonl` (208 Q&A: python/js/linux/security/networking), `so_python.jsonl`, `so_security.jsonl`, `web_docs.txt`. Chạy lần lượt các cell dưới đây trên GPU Colab (T4 miễn phí đủ cho tiny; small cần bật grad_checkpoint). Quota SE API reset mỗi ngày — muốn lấy thêm thì chạy lại `fetch_all.py` trên Colab hoặc gắn API key miễn phí (stackapps.com).

In [ ]:
!pip -q install torch numpy pyyaml tqdm tensorboard pytest requests datasets
!python -m pytest test/ -q  # kỳ vọng: pass hết (gồm test GQA + KV-cache)

In [ ]:
# 1. Build .bin từ data THẬT (SO python + security + docs). Muốn thêm data: chạy collector rồi thêm file vào --input
!python data_prime/stackoverflow_elite/api_prime.py --out data_prime/raw/so_elite.jsonl --max 300
!python data_prime/wiki_elite/hf_stream.py --max-en 8000 --max-vi 4000
!python data_prime/build/merge_prime.py --out data/raw/prime_all.jsonl
!python scripts/prepare_data.py --input data/raw/prime_all.jsonl,data/raw/train.txt --train-out data/processed/train.bin --val-out data/processed/val.bin --tok-out data/tokenizer/byte.json

In [ ]:
# 2a. Pretrain TINY smoke test (~3M params, vài phút trên T4)
!python -m training.train --config config/model/tiny.yaml
# 2b. Pretrain SMALL (~30M, GQA 8q/2kv, ctx 1024) — chạy khi tiny đã loss giảm
# !python -m training.train --config config/model/small.yaml

In [ ]:
!ls experiments/v0.1/checkpoints/
!python -m evaluation.evaluate --config config/model/tiny.yaml --ckpt experiments/v0.1/checkpoints/step_005000.pt

In [ ]:
# 3. SFT v1.1 defensive
!python -m training.sft --config config/model/tiny.yaml --data data/sft/security_sft.jsonl --base-ckpt experiments/v0.1/checkpoints/step_005000.pt --out experiments/v1.1/sft.pt --max-steps 60 --lr 1e-5

In [ ]:
!python -m security.eval.safety_eval --config config/model/tiny.yaml --ckpt experiments/v1.1/sft.pt
!python -m inference.generate --ckpt experiments/v1.1/sft.pt --prompt "<user>How do I secure my home WiFi?</user>" --max-new-tokens 80 --temperature 0.0

In [ ]:
# 3c. Branched generation: sample 4 nhánh, tự chấm (format + ít lặp), trả nhánh tốt nhất
# !python -m inference.generate --ckpt experiments/v1.1/sft.pt --prompt "<user>How do I secure my home WiFi?</user>" --max-new-tokens 80 --temperature 0.8 --top-p 0.9 --best-of 4

In [ ]:
# 3b. DPO preference (v1.2 alignment): học chosen > rejected. Chạy sau SFT, trước GRPO.
# !python -m training.dpo --config config/model/tiny.yaml --data data/sft/security_prefs.jsonl --base-ckpt experiments/v1.1/sft.pt --out experiments/v1.2/dpo.pt --steps 40
# !python -m security.eval.pref_eval --config config/model/tiny.yaml --ckpt experiments/v1.2/dpo.pt

In [ ]:
# 4. RL suy nghĩ nhiều bước (GRPO): SFT reasoning trước rồi RL. Chạy sau khi SFT xong.
# SFT warmup với think traces:
# !python -m training.sft --config config/model/tiny.yaml --data data/sft/security_reasoning.jsonl --base-ckpt experiments/v1.1/sft.pt --out experiments/v1.3/sft_reason.pt --max-steps 60 --lr 1e-5
# (khuyến nghị: base từ DPO experiments/v1.2/dpo.pt để có pipeline SFT → DPO → GRPO)
# GRPO (G=4, ~vài phút trên T4 với tiny):
# !python -m training.grpo --config config/model/tiny.yaml --data data/sft/security_reasoning.jsonl --base-ckpt experiments/v1.3/sft_reason.pt --out experiments/v1.3/grpo.pt --steps 50 --G 4
# !python -m security.eval.reasoning_eval --config config/model/tiny.yaml --ckpt experiments/v1.3/grpo.pt

### Lấy thêm data (chạy trên Colab, tuân thủ API + CC-BY-SA)
`!python data_collectors/stackoverflow/collect_so.py --tag python --pages 5 --out data/raw/so_python2.jsonl` — rồi thêm file vào `--input` ở cell 1 và build lại `.bin`.